# RE_202 · Qui acabarà comprant?

**Optativa d'Aprenentatge automàtic · Bloc 2 — Supervisat · Dificultat 2 · durada: 1 o 2 sessions de 2 h**

Treballes a l'equip de dades d'una botiga en línia. Màrqueting no vol una caixa negra que
«endevini»: vol un model que **expliqui** quins senyals de la navegació empenyen una sessió cap a
la compra, i que doni una **probabilitat** que es pugui posar en un tauler.

**La pregunta:** amb una regressió logística, quines variables pesen a favor i en contra de la compra, quant,
i com arriba el model a la probabilitat d'una sessió concreta?

## Les dades

- **Font:** *Online Shoppers Purchasing Intention* (Sakar i Kastro, 2018; UCI), a OpenML amb
  `data_id=46947`. Llicència CC BY 4.0.
- **Cada fila** és **una sessió** d'un visitant en una botiga en línia durant un any: 12.330 files i
  18 columnes. No hi ha cap identificador de persona.
- Documentació: <https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset>

### Diccionari de columnes

| Columna | Què és |
| --- | --- |
| `Revenue` | **objectiu**: si la sessió va acabar en compra (`True` / `False`, com a text) |
| `Administrative`, `Informational`, `ProductRelated` | nombre de pàgines de cada tipus vistes a la sessió (gestió del compte, informació, productes) |
| `Administrative_Duration`, `Informational_Duration`, `ProductRelated_Duration` | temps passat a cada tipus de pàgina, en segons |
| `BounceRates` | taxa de rebot mitjana de les pàgines vistes (Google Analytics): proporció de visites que entren per aquella pàgina i se'n van sense fer res més |
| `ExitRates` | taxa de sortida mitjana de les pàgines vistes: proporció de visites en què aquella pàgina és l'última |
| `PageValues` | valor mitjà de les pàgines vistes, segons Google Analytics. Llegeix a la documentació com es calcula |
| `SpecialDay` | proximitat a un dia especial (Sant Valentí, Dia de la Mare...), de 0 a 1 |
| `Month` | mes de la sessió (text) |
| `OperatingSystems`, `Browser`, `Region`, `TrafficType` | codis del sistema operatiu, el navegador, la regió i l'origen del trànsit |
| `VisitorType` | `Returning_Visitor`, `New_Visitor` o `Other` |
| `Weekend` | si la sessió és en cap de setmana |

### Com s'han recollit, i què cal tenir present

- Les dades són d'una botiga real, recollides amb Google Analytics durant un any. Cada sessió és d'un
  usuari diferent, perquè no hi hagi tendències d'una sola persona.
- Les mètriques de Google Analytics es calculen sobre totes les visites del web, no només sobre
  aquesta sessió.

## Què has d'investigar

Construeix el model que demana màrqueting i explica'l. Abans, fixa la **línia de sortida**:
quina precisió té un model que mai no diu «compra»? Decideix quines columnes són números de
debò i quines són codis. Entrena una **regressió logística** amb les numèriques escalades, compara
la precisió amb la línia de sortida, i llegeix-ne els pesos: quines variables empenyen cap a la
compra, quines en contra, i quant (tradueix el pes més gran a *odds*).

La teoria ha d'aterrar al codi: calcula **a mà** la probabilitat de compra de totes les sessions
de test, primer $z = Xw + b$ i després la sigmoide, i comprova-la contra `predict_proba`.
Descompon la $z$ d'una sessió columna per columna. Implementa la **log-loss** amb NumPy i
comprova-la amb `log_loss`; compara-la amb la d'un model que dona a tothom la mateixa
probabilitat.

**Per anar més enllà** (no és obligatori, i no hi ha una sola manera de fer-ho):

- Si afegeixes les categòriques (mes, tipus de visitant...), millora la log-loss? Què faria el model amb una sessió d'un mes que no ha vist mai?
- Si treus la variable amb més pes, quant empitjora el model? Aquesta variable, la sabries a mitja sessió, quan voldries oferir un descompte?

**Pistes per pensar** (no són la resposta):

- Si una sola variable fa gairebé tota la feina, abans de celebrar-ho pregunta't quan es coneix el seu valor.
- La precisió i la log-loss poden no anar juntes. Si el model amb prou feines supera la línia de sortida en precisió, mira l'altra mesura abans de dir que no serveix.

## Exemples per inspirar-te

Quaderns del curs on es fan servir les eines que et caldran. Mira com es treballa, no hi busquis
la resposta: les dades d'aquest repte són unes altres.

- [ML_04 · regressió logística](../01_teoria/ML_04_regressio_logistica.ipynb) §2-§4: la regressió logística, la sigmoide i els pesos.
- [MA_01 · àlgebra lineal](../03_matematiques/MA_01_algebra_lineal.ipynb) §7 (un model és un vector i una multiplicació) i [MA_02 · descens del gradient](../03_matematiques/MA_02_descens_gradient.ipynb) §5 (la log-loss).
- [EX_00 · caixa d'eines](../02_practica/EX_00_caixa_eines.ipynb) §2 (`StandardScaler` i `fit` contra `transform`), §4, §6 i §7 (`OneHotEncoder`, `ColumnTransformer`, `DummyClassifier`).
- [EX_02 · Breast cancer](../02_practica/EX_02_cancer.ipynb): la precisió que enganya.

## Lliurament

Lliura aquest quadern **executat**, sense errors, amb la teva anàlisi: cel·les de codi i, entre elles, cel·les de text on expliques què has fet i per què. Cada decisió que prenguis (què fas amb un buit, quina columna treus, quin model tries) ha de tenir la seva línia de justificació. Acaba amb una **conclusió** de tres a cinc frases, escrita per tu, que respongui l'encàrrec amb els números que has trobat.

> **Quan cridar en Xavi:** per a dubtes de **criteri**, no de sintaxi (per a la sintaxi hi ha la
> documentació i els exemples). Per exemple: «El pes més gran és tan gran que em fa sospitar. Puc fer-me'l meu?» «Aquesta variable, la puc fer servir o és fer trampa?»

La cel·la següent carrega les dades. A partir d'aquí el quadern és teu: afegeix-hi totes les
cel·les de codi i de text que necessitis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=46947, as_frame=True).frame
print(df.shape)
df.head()